In [0]:
%run "../../commons/commons_imports"

In [0]:
df_municipio_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_MUNICIPIO,
    format="delta"
).select(
    col("ANO_REFERENCIA"),
    col("CO_UF"),
    col("SG_UF"),
    col("REGIAO"),
    col("CO_MUNICIPIO"),
    col("NO_MUNICIPIO"),
    col("NO_MUNICIPIO_UF"),
    col("ID_TIPO_REDE"),
    col("DS_TIPO_REDE"),
    col("PC_ALUNO_ALFABETIZADO"),
    col("VL_MEDIA_LP"),
    col("FAIXA_MEDIA_LP"),
    col("FAIXA_ALFABETIZACAO"),
    col("PC_ALUNO_NIVEL_0_LP"),
    col("PC_ALUNO_NIVEL_1_LP"),
    col("PC_ALUNO_NIVEL_2_LP"),
    col("PC_ALUNO_NIVEL_3_LP"),
    col("PC_ALUNO_NIVEL_4_LP"),
    col("PC_ALUNO_NIVEL_5_LP"),
    col("PC_ALUNO_NIVEL_6_LP"),
    col("PC_ALUNO_NIVEL_7_LP"),
    col("PC_ALUNO_NIVEL_8_LP"),
).filter(col('ID_TIPO_REDE').isin([3]))

In [0]:
window = Window.partitionBy("CO_MUNICIPIO").orderBy(col("ANO_REFERENCIA").desc())

df_metas_municipio = (
    read(
        base_path=SILVER_PATH,
        table_name=METAS_MUNICIPIO,
        format="delta"
    )
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
    .select(
        col("CO_MUNICIPIO").alias("CO_MUNICIPIO_META"),
        col("META_ALFABETIZACAO_2024"),
        col("META_ALFABETIZACAO_2025"),
        col("META_ALFABETIZACAO_2026"),
        col("META_ALFABETIZACAO_2027"),
        col("META_ALFABETIZACAO_2028"),
        col("META_ALFABETIZACAO_2029"),
        col("META_ALFABETIZACAO_2030"),      
        col("IN_POSSUI_META")
    )
)

In [0]:
df_metas_resultados_municipio = df_municipio_silver.join(
    df_metas_municipio,
    how='left',
    on=[(df_municipio_silver.CO_MUNICIPIO == df_metas_municipio.CO_MUNICIPIO_META)]
).drop(
    'CO_MUNICIPIO_META'
)

In [0]:
 df_metas_resultados_municipio_enriquecido = (

    df_metas_resultados_municipio

    # =====================================================
    # Buscar a meta dinamicamente de acordo com o ano da avaliação
    # =====================================================

    .withColumn(
        "META_ANO_AVALIACAO",
        when(col("ANO_REFERENCIA") == 2024, col("META_ALFABETIZACAO_2024"))
        .when(col("ANO_REFERENCIA") == 2025, col("META_ALFABETIZACAO_2025"))
        .when(col("ANO_REFERENCIA") == 2026, col("META_ALFABETIZACAO_2026"))
        .when(col("ANO_REFERENCIA") == 2027, col("META_ALFABETIZACAO_2027"))
        .when(col("ANO_REFERENCIA") == 2028, col("META_ALFABETIZACAO_2028"))
        .when(col("ANO_REFERENCIA") == 2029, col("META_ALFABETIZACAO_2029"))
        .when(col("ANO_REFERENCIA") == 2030, col("META_ALFABETIZACAO_2030"))
        .otherwise(None)
    )

    # =====================================================
    # Quanto faltou ou quanto passou da meta
    # =====================================================
    .withColumn(
        "DIF_META_ALFABETIZACAO",
        round(
            col("PC_ALUNO_ALFABETIZADO") - col("META_ANO_AVALIACAO"),
            2
        )
    )

    # =====================================================
    # Flag: Bateu a meta? (1 = Sim, 0 = Não)
    # Regra: Se a meta for nula, verifica se PC_ALUNO_ALFABETIZADO > 80 ("Sim" ou "Não")
    # Caso contrário, verifica se bateu a META_ANO_AVALIACAO ("Sim" ou "Não")
    # =====================================================

    .withColumn(
        "IN_META_ATINGIDA",
        when(col("META_ANO_AVALIACAO").isNull(),
            when(col("PC_ALUNO_ALFABETIZADO") > 80, lit("Sim")).otherwise(lit("Não"))).
        otherwise(
            when(col("PC_ALUNO_ALFABETIZADO") >= col("META_ANO_AVALIACAO"), lit("Sim")).otherwise(lit("Não"))
        )
    )

    # =====================================================
    # Distância para o objetivo final de 2030 (>80% de alfabetização)
    # =====================================================

    .withColumn(
        "DISTANCIA_META_2030",
        round(
            col("PC_ALUNO_ALFABETIZADO") - col("META_ALFABETIZACAO_2030"), 2)
    )

    # =====================================================
    # Flag: Bateu a meta de 2030? (1 = Sim, 0 = Não)
    # =====================================================

    .withColumn(
        "IN_META_2030_ATINGIDA",
        when(col("DISTANCIA_META_2030") >= 0, "Sim").otherwise("Não")
    )

)

In [0]:
df_metas_resultados_municipio_enriquecido = (df_metas_resultados_municipio_enriquecido
    .select(
        col("ANO_REFERENCIA"),
        col("CO_UF"),
        col("SG_UF"),
        col("REGIAO"),
        col("CO_MUNICIPIO"),
        col("NO_MUNICIPIO"),
        col("NO_MUNICIPIO_UF"),
        col("ID_TIPO_REDE"),
        col("DS_TIPO_REDE"),
        col("VL_MEDIA_LP"),
        col("FAIXA_MEDIA_LP"),
        col("PC_ALUNO_ALFABETIZADO"),    
        col("META_ANO_AVALIACAO"),
        col("DIF_META_ALFABETIZACAO"),
        col("IN_META_ATINGIDA"),
        col("DISTANCIA_META_2030"),
        col("IN_META_2030_ATINGIDA")
    )
)


In [0]:
df_analise_niveis = ( df_metas_resultados_municipio

    # =====================================================
    # Agrupamento de alunos em situação crítica (Níveis 0 e 1)
    # =====================================================

    .withColumn(
        "PC_PERFIL_EXTREMA_DEFASAGEM",
        round(col("PC_ALUNO_NIVEL_0_LP") + col("PC_ALUNO_NIVEL_1_LP"), 2)
    )

    # =====================================================
    # Agrupamento de alunos em desenvolvimento (Níveis 2 e 3)
    # =====================================================

    .withColumn(
        "PC_PERFIL_EM_DESENVOLVIMENTO",
        round(col("PC_ALUNO_NIVEL_2_LP") + col("PC_ALUNO_NIVEL_3_LP"), 2)
    )

    # =====================================================
    # Agrupamento de alunos no limítrofe (Níveis 4 - 725 a 750)
    # =====================================================

    .withColumn(
        "PC_PERFIL_LIMITROFE",
         round(col("PC_ALUNO_NIVEL_4_LP") + col("PC_ALUNO_NIVEL_1_LP"), 2)
    )

    # =====================================================
    # Agrupamento de alunos em nível avançado/adequado (Níveis 5 e 6)
    # =====================================================

    .withColumn(
        "PC_PERFIL_AVANCADO",
        round(col("PC_ALUNO_NIVEL_5_LP") + col("PC_ALUNO_NIVEL_6_LP"), 2)
    )
    # =====================================================
    # Agrupamento de alunos em excelência (Níveis 7 e 8)
    # =====================================================

    .withColumn(
        "PC_TAXA_EXCELENCIA",
        round(col("PC_ALUNO_NIVEL_7_LP") + col("PC_ALUNO_NIVEL_8_LP"), 2)
    )

    # =====================================================
        # ÍNDICE DE POLARIZAÇÃO (Desigualdade)
        # Compara a proporção de alunos na excelência versus na extrema defasagem.
        # Usamos + 0.01 no divisor para evitar erro de divisão por zero.
        # Se o valor for próximo a 1, significa que há tantos alunos excelentes quanto em extrema defasagem (alta desigualdade).
        # Quanto mais próximo de 0 pior é o rendimento do município (mais alunos em defasagem). Quanto maior que 1 maior é o rendimento do município (mais alunos em excelência).  
    # =====================================================
    .withColumn(
        "INDICE_POLARIZACAO",
        round(
            col("PC_TAXA_EXCELENCIA") / (col("PC_PERFIL_EXTREMA_DEFASAGEM") + 0.01), 2
        )
    )

    # =====================================================
    # Índice de Risco Estrutural (Mede se a defasagem está concentrada no pior nível possível)
    # Quanto maior esse índice (mais próximo de 1 ou 100%), pior a situação dos alunos não alfabetizados
    # =====================================================
    .withColumn(
    "INDICE_RISCO_ESTRUTURAL",
    when(
        (col("PC_PERFIL_EXTREMA_DEFASAGEM") + col("PC_PERFIL_EM_DESENVOLVIMENTO")) > 0,
            round(
            (
                (col("PC_ALUNO_NIVEL_0_LP") * 4) + 
                (col("PC_ALUNO_NIVEL_1_LP") * 3) + 
                (col("PC_ALUNO_NIVEL_2_LP") * 2) +
                (col("PC_ALUNO_NIVEL_3_LP") * 1)
            ) / 400.0, 4 # Arredondando para 4 casas decimais para precisão
            )
        ).otherwise(0)  
    )
)

In [0]:
df_analise_niveis = (df_analise_niveis
    .select(
        col("ANO_REFERENCIA"),
        col("CO_UF"),
        col("SG_UF"),
        col("REGIAO"),
        col("CO_MUNICIPIO"),
        col("NO_MUNICIPIO"),
        col("NO_MUNICIPIO_UF"),
        col("ID_TIPO_REDE"),
        col("DS_TIPO_REDE"),
        col("VL_MEDIA_LP"),
        col("FAIXA_MEDIA_LP"),
        col("PC_ALUNO_ALFABETIZADO"),    
        col("PC_ALUNO_NIVEL_0_LP"),
        col("PC_ALUNO_NIVEL_1_LP"),
        col("PC_ALUNO_NIVEL_2_LP"),
        col("PC_ALUNO_NIVEL_3_LP"),
        col("PC_ALUNO_NIVEL_4_LP"),
        col("PC_ALUNO_NIVEL_5_LP"),
        col("PC_ALUNO_NIVEL_6_LP"),
        col("PC_ALUNO_NIVEL_7_LP"),
        col("PC_ALUNO_NIVEL_8_LP"),
        col("PC_PERFIL_EXTREMA_DEFASAGEM"),
        col("PC_PERFIL_EM_DESENVOLVIMENTO"),
        col("PC_PERFIL_LIMITROFE"),
        col("PC_PERFIL_AVANCADO"),
        col("PC_TAXA_EXCELENCIA"),
        col("INDICE_POLARIZACAO"),
        col("INDICE_RISCO_ESTRUTURAL")
    )
)

In [0]:
write_delta(
    df=df_metas_resultados_municipio_enriquecido,
    base_path=GOLD_PATH,
    table_name=METAS_MUNICIPIO_META_VS_RESULTADO,
    write_mode="overwrite"
)

write_delta(
    df=df_analise_niveis,
    base_path=GOLD_PATH,
    table_name=ANALISE_NIVEIS_MUNICIPIO,
    write_mode="overwrite"
)